#### **Попробуем применить Лассо-регрессию к данным из кик-старта.**

_Считаем данные._

In [1]:
import pandas as pd
import numpy as np

x = pd.read_csv('../../5. Обобщаяющая способность/Практика с кик старта/X.csv')
y = pd.read_csv('../../5. Обобщаяющая способность/Практика с кик старта/Y.csv')
x = x.drop('Название', axis=1)
x

,Категория,Цель в долларах,Срок,Год публикации,Close_brent,CAD,CHF,DKK,EUR,GBP,...,Design,Fashion,Film & Video,Food,Journalism,Music,Photography,Publishing,Technology,Theater
0,6035.989239,1000.00,39,2009,34.41,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
1,3591.033473,80000.00,87,2009,34.41,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
2,3661.424550,20.00,8,2009,34.41,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,4321.245721,99.00,79,2009,34.41,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
4,6035.989239,1900.00,28,2009,34.41,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
331670,7635.064778,35.98,2,2017,34.41,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
331671,38415.722876,271.03,4,2017,34.41,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
331672,6098.303122,200.00,3,2017,34.41,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
331673,38415.722876,250.00,1,2017,34.41,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


_Сначала получим индексы для разбиения выборки на трейн и тест._

In [30]:
from sklearn.model_selection import KFold
#создадим правило разбиения строк
kf = KFold(n_splits=10, shuffle=True, random_state=33)
#создадим индексы для разбиения строк
indexes_split = kf.split(x)

_Обучим Лассо-регрессию и применим к ней кросс-валидацию._
* _делим данные_
* _масшатабируем только на трейне_
* _обучаем_
* _сохраняем результаты_

In [31]:
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
losses_test = []
losses_train = []

#чтобы обращаться к X и Y по индексам строк
#переведем их в np.array
x_np = x.to_numpy()
y_np = y.values

#проведем обучения для каждого разбиения
for tr_i, ts_i in indexes_split:
    #разделим данные -=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
    #тренировочная
    x_train = x_np[tr_i]
    y_train = y_np[tr_i]
    #тестовая
    x_test = x_np[ts_i]
    y_test = y_np[ts_i]

    #масштабируем признаки -=-=-=-=-=-=-=-=-=-=-=-=-=-=
    scaler = StandardScaler() #объект для стандартизации
    #находим среднее и стандартное отклонение
    scaler.fit(x_train)
    #изменяем данные
    x_train_std = scaler.transform(x_train)
    x_test_std = scaler.transform(x_test)

    #масштабируем ответы -=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
    scaler = StandardScaler() #новый объект
    scaler.fit(y_train)
    y_train_std = scaler.transform(y_train).ravel()
    y_test_std = scaler.transform(y_test).ravel()

    #обучим модель -=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
    model = Lasso(alpha=10)
    model.fit(x_train_std, y_train_std)

    #ошибка на трейне -=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
    error_tr = np.mean((model.predict(x_train_std) - y_train_std)**2)
    losses_train.append(error_tr**(1/2))

    #ошибка на тесте -=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
    error_ts = np.mean((model.predict(x_test_std) - y_test_std)**2)
    losses_test.append(error_ts**(1/2))

_Оценим ошибку._

In [32]:
int(np.mean(losses_test)), int(np.mean(losses_train))

(0, 1)